In [0]:
from pyspark.sql import functions as F
spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")  
BASE = "/Volumes/workspace/landing/inputs"

## Ingestão dos CSVs para a Bronze

Esta célula lê cada um dos cinco CSVs do volume e grava como uma tabela Delta na camada Bronze. Os dados entram como texto (sem inferir tipos), para não alterar nada do conteúdo original. As opções `multiLine` e `escape` evitam que sinopses e comentários com aspas ou quebras de linha sejam cortados no meio. Cada tabela recebe a coluna `ingestion_datetime`, com o momento exato da gravação, e a escrita é feita em modo append.

In [0]:
# inferSchema=False mantém tudo como string (nenhuma alteração de conteúdo).
# multiLine/escape evitam quebrar linhas de sinopses e comentários com aspas ou quebras de linha.
arquivos = {
    "movies_info_TMDB_IMDB.csv":       "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":     "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":  "tb_credits_and_tags",
    "movies_reviews.csv":              "tb_movies_reviews",
}

for arq, tabela in arquivos.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .option("multiLine", True)
          .option("quote", '"').option("escape", '"')
          .csv(f"{BASE}/{arq}"))
    # Timestamp do momento da inserção na Bronze
    df = df.withColumn("ingestion_datetime", F.current_timestamp())
    (df.write.format("delta").mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(f"bronze.{tabela}"))
    print(tabela, "->", df.count(), "linhas")

tb_movies_info -> 106596 linhas
tb_movies_financials -> 106165 linhas
tb_movies_metrics -> 103072 linhas
tb_credits_and_tags -> 105608 linhas
tb_movies_reviews -> 32412 linhas


## Ingestão da cotação do dólar (API do Banco Central)

Aqui buscamos a cotação do dólar na API PTAX do Banco Central e gravamos em `bronze.tb_cotacao_dolar`. As datas de início e fim são parâmetros (widgets) no formato MM-DD-AAAA, e o padrão é consultar os últimos 7 dias. O código monta a URL, faz a requisição e converte o resultado em um DataFrame com a data e a cotação de compra. Se a API não devolver nenhum dado, a célula para com uma mensagem de erro pedindo para ampliar o período. Por fim, adicionamos o `ingestion_datetime` e gravamos em Delta com append.

In [0]:
import requests
from datetime import datetime, timedelta

# datas no formato MM-DD-AAAA, padrão = últimos 7 dias corridos
dbutils.widgets.text("data_inicio", (datetime.now() - timedelta(days=7)).strftime("%m-%d-%Y"))
dbutils.widgets.text("data_fim", datetime.now().strftime("%m-%d-%Y"))
ini = dbutils.widgets.get("data_inicio")
fim = dbutils.widgets.get("data_fim")

url = ("https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
       "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
       f"?@dataInicial='{ini}'&@dataFinalCotacao='{fim}'"
       "&$select=dataHoraCotacao,cotacaoCompra&$format=json")

def gerar_mock(ini, fim):
    """Mock usado só se a API estiver indisponível.
    Mantém o mesmo formato da resposta real (dataHoraCotacao e cotacaoCompra) e, como a API,
    não gera cotação em fins de semana. Os valores são fixos e semelhantes aos reais."""
    valores = [5.1803, 5.2073, 5.2232, 4.9853, 4.9692]
    dia = datetime.strptime(ini, "%m-%d-%Y")
    limite = datetime.strptime(fim, "%m-%d-%Y")
    mock, i = [], 0
    while dia <= limite:
        if dia.weekday() < 5:  # segunda a sexta
            mock.append({"dataHoraCotacao": dia.strftime("%Y-%m-%d") + " 13:05:00.000",
                         "cotacaoCompra": valores[i % len(valores)]})
            i += 1
        dia += timedelta(days=1)
    return mock

try:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    dados = resp.json()["value"]
    print("Cotação obtida da API do Banco Central.")
except Exception as e:
    print(f"API indisponível ({type(e).__name__}). Usando dados mock.")
    dados = gerar_mock(ini, fim)

if not dados:
    raise ValueError("Sem cotações no período. Amplie as datas nos widgets.")

df_cot = spark.createDataFrame(
    [(d["dataHoraCotacao"], float(d["cotacaoCompra"])) for d in dados],
    "dataHoraCotacao string, cotacaoCompra double"
).withColumn("ingestion_datetime", F.current_timestamp())

df_cot.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
display(spark.table("bronze.tb_cotacao_dolar"))

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-30 13:11:44.783262,5.1803,2026-10-07T00:42:26.024Z
2026-10-01 13:10:35.4469,5.2073,2026-10-07T00:42:26.024Z
2026-10-02 13:03:16.256632,5.2232,2026-10-07T00:42:26.024Z
2026-10-05 13:07:36.558602,4.9853,2026-10-07T00:42:26.024Z
2026-10-06 13:03:21.267287,4.9692,2026-10-07T00:42:26.024Z


## Conferência da Bronze

Esta célula mostra quantas linhas cada tabela da Bronze tem depois das cargas. Serve para conferir se todas foram criadas e se nenhuma foi carregada duas vezes por engano.

In [0]:
for t in ["tb_movies_info","tb_movies_financials","tb_movies_metrics","tb_credits_and_tags","tb_movies_reviews","tb_cotacao_dolar"]:
    print(t, spark.table(f"bronze.{t}").count())

tb_movies_info 213192
tb_movies_financials 212330
tb_movies_metrics 103072
tb_credits_and_tags 105608
tb_movies_reviews 32412
tb_cotacao_dolar 5
